# Painter classification

Run this on a Colab GPU runtime. The first cell mounts Google Drive, installs the libraries, and downloads the course images (`famous_paintings.zip`, id `1-0d315aj7Ai8NNqat65XDvaOcHDcHiUD`).

Torchvision provides ResNet-18, ResNet-34, and ResNet-152. ResNet-152 is used for the ResNet-150 slot. After each model finishes, two weight files are written to `MyDrive/famous_paintings_weights`: the best validation F1 and the best validation accuracy. That is six files in total.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/famous_paintings_weights")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("weights folder:", DRIVE_DIR)

from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/famous_paintings_weights")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("weights folder:", DRIVE_DIR)

# Colab already has a matched torch, numpy, scipy, and scikit-learn.
# Upgrading numpy in this session breaks sklearn, so only gdown is installed.
!pip -q install gdown

import numpy
import torch
import torchvision
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

print("numpy", numpy.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("torchvision", torchvision.__version__)
print("sklearn import ok")

!gdown 1-0d315aj7Ai8NNqat65XDvaOcHDcHiUD
!unzip -q -o famous_paintings.zip



In [ ]:
import shutil
from pathlib import Path

DATA_DIR = Path("data")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

flat_images = [
    path
    for path in DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
]

if flat_images:
    painters = sorted({path.stem.rsplit("_", 1)[0] for path in flat_images})
    painter_to_idx = {name: index for index, name in enumerate(painters)}
    for index in painter_to_idx.values():
        (DATA_DIR / str(index)).mkdir(exist_ok=True)
    for path in flat_images:
        painter = path.stem.rsplit("_", 1)[0]
        shutil.move(path, DATA_DIR / str(painter_to_idx[painter]) / path.name)
else:
    painter_to_idx = {}
    for class_dir in DATA_DIR.iterdir():
        if not class_dir.is_dir() or not class_dir.name.isdigit():
            continue
        sample = next(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in IMAGE_SUFFIXES
        )
        painter_to_idx[sample.stem.rsplit("_", 1)[0]] = int(class_dir.name)

idx_to_painter = dict(sorted((index, name) for name, index in painter_to_idx.items()))
print(idx_to_painter)
print(f"painters: {len(idx_to_painter)}")


In [ ]:
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms

SEED = 42


def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)


seed_everything(SEED)
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

DATA_DIR = "data"
BATCH_SIZE = 32
IMAGE_SIZE = 224
MAX_EPOCHS = 200
INITIAL_LR = 1e-3
LR_PATIENCE = 10
LR_FACTOR = 0.5
EARLY_STOP_PATIENCE = 45
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train_transform = transforms.Compose(
    [
        transforms.Resize(248),
        transforms.RandomAffine(
            degrees=8,
            scale=(0.94, 1.06),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = transforms.Compose(
    [
        transforms.Resize(248),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class NumericImageFolder(datasets.ImageFolder):
    """Class index is the numeric folder name."""

    def find_classes(self, directory):
        classes = [
            entry.name
            for entry in os.scandir(directory)
            if entry.is_dir() and entry.name.isdigit()
        ]
        classes.sort(key=int)
        return classes, {name: int(name) for name in classes}


class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]
        return self.transform(image), label


def painter_names(root):
    names = {}
    for class_dir in Path(root).iterdir():
        if not class_dir.is_dir() or not class_dir.name.isdigit():
            continue
        sample = next(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        names[int(class_dir.name)] = sample.stem.rsplit("_", 1)[0].replace("_", " ")
    return [names[i] for i in range(len(names))]


base_dataset = NumericImageFolder(DATA_DIR)
labels = painter_names(DATA_DIR)
num_classes = len(base_dataset.classes)

train_idx, val_idx = train_test_split(
    np.arange(len(base_dataset)),
    test_size=0.2,
    random_state=SEED,
    stratify=base_dataset.targets,
)
train_set = TransformSubset(Subset(base_dataset, train_idx), train_transform)
val_set = TransformSubset(Subset(base_dataset, val_idx), val_transform)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    worker_init_fn=seed_worker,
)

print(f"images: {len(base_dataset)}   classes: {num_classes}")
print(f"train: {len(train_set)}   validation: {len(val_set)}")
print(labels)

In [ ]:
METRIC_NAMES = ("loss", "accuracy", "precision", "recall", "f1")
SCORE_ARGS = dict(average="macro", labels=list(range(num_classes)), zero_division=0)


def make_model(name):
    builders = {
        "resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
        "resnet34": (models.resnet34, models.ResNet34_Weights.DEFAULT),
        "resnet152": (models.resnet152, models.ResNet152_Weights.DEFAULT),
    }
    builder, weights = builders[name]
    model = builder(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, **SCORE_ARGS),
        "recall": recall_score(y_true, y_pred, **SCORE_ARGS),
        "f1": f1_score(y_true, y_pred, **SCORE_ARGS),
    }


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total = 0
    truths = []
    preds = []
    for images, batch_labels in loader:
        images = images.to(device)
        batch_labels = batch_labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = torch.nn.functional.cross_entropy(logits, batch_labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * batch_labels.size(0)
        total += batch_labels.size(0)
        truths.append(batch_labels.cpu())
        preds.append(logits.argmax(dim=1).cpu())
    metrics = classification_metrics(torch.cat(truths).numpy(), torch.cat(preds).numpy())
    metrics["loss"] = total_loss / total
    return metrics


def cpu_state(state):
    return {key: value.detach().cpu() for key, value in state.items()}


def train_model(model, name):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LR)
    history = {split: {metric: [] for metric in METRIC_NAMES} for split in ("train", "val")}
    best_f1 = -1.0
    best_f1_epoch = 0
    best_f1_state = cpu_state(model.state_dict())
    best_acc = -1.0
    best_acc_epoch = 0
    best_acc_state = cpu_state(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        lr = optimizer.param_groups[0]["lr"]
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)
        for metric in METRIC_NAMES:
            history["train"][metric].append(train_metrics[metric])
            history["val"][metric].append(val_metrics[metric])
        print(
            f"{name}  epoch {epoch:03d}/{MAX_EPOCHS}  lr {lr:.1e}  "
            f"train loss {train_metrics['loss']:.3f} acc {train_metrics['accuracy']:.3f} "
            f"prec {train_metrics['precision']:.3f} rec {train_metrics['recall']:.3f} f1 {train_metrics['f1']:.3f}  "
            f"val loss {val_metrics['loss']:.3f} acc {val_metrics['accuracy']:.3f} "
            f"prec {val_metrics['precision']:.3f} rec {val_metrics['recall']:.3f} f1 {val_metrics['f1']:.3f}  "
            f"({time.time() - started:.0f}s)",
            flush=True,
        )

        improved_f1 = val_metrics["f1"] > best_f1
        if improved_f1:
            best_f1 = val_metrics["f1"]
            best_f1_epoch = epoch
            best_f1_state = cpu_state(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement % LR_PATIENCE == 0:
                optimizer.param_groups[0]["lr"] *= LR_FACTOR
                print(
                    f"{name}  no val f1 improvement for {epochs_without_improvement} epochs, "
                    f"lr -> {optimizer.param_groups[0]['lr']:.1e}",
                    flush=True,
                )

        if val_metrics["accuracy"] > best_acc:
            best_acc = val_metrics["accuracy"]
            best_acc_epoch = epoch
            best_acc_state = cpu_state(model.state_dict())

        if not improved_f1 and epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"{name}  stopping at epoch {epoch}", flush=True)
            break

    f1_path = DRIVE_DIR / f"{name}_best_f1.pth"
    acc_path = DRIVE_DIR / f"{name}_best_accuracy.pth"
    torch.save(best_f1_state, f1_path)
    torch.save(best_acc_state, acc_path)
    print(
        f"{name}  best val f1 {best_f1:.3f} at epoch {best_f1_epoch}  -> {f1_path}",
        flush=True,
    )
    print(
        f"{name}  best val accuracy {best_acc:.3f} at epoch {best_acc_epoch}  -> {acc_path}",
        flush=True,
    )
    model.load_state_dict(best_f1_state)
    return history


def predict(model, loader):
    model.eval()
    probabilities = []
    truths = []
    with torch.no_grad():
        for images, batch_labels in loader:
            logits = model(images.to(device))
            probabilities.append(logits.softmax(dim=1).cpu())
            truths.append(batch_labels)
    return torch.cat(probabilities).numpy(), torch.cat(truths).numpy()


def best_f1_threshold(probabilities, y_true):
    """One shared cutoff: a painter is selected when its probability is at least t."""
    best_threshold = 0.5
    best_score = -1.0
    curve = []
    for threshold in np.round(np.arange(0.05, 1.0, 0.05), 2):
        scores = [
            f1_score(y_true == class_id, probabilities[:, class_id] >= threshold, zero_division=0)
            for class_id in range(num_classes)
        ]
        score = float(np.mean(scores))
        curve.append(score)
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold, best_score, np.round(np.arange(0.05, 1.0, 0.05), 2), curve


def plot_training(name, history):
    titles = {
        "loss": "loss",
        "accuracy": "accuracy",
        "precision": "precision (macro)",
        "recall": "recall (macro)",
        "f1": "f1 (macro)",
    }
    fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
    for ax, metric in zip(axes.ravel(), METRIC_NAMES):
        epochs = np.arange(1, len(history["train"][metric]) + 1)
        ax.plot(epochs, history["train"][metric], label="train")
        ax.plot(epochs, history["val"][metric], label="validation")
        ax.set_title(titles[metric])
        ax.set_xlabel("epoch")
        ax.legend()
    axes.ravel()[-1].axis("off")
    fig.suptitle(name)
    fig.tight_layout()
    plt.show()


def plot_confusion(name, y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    row_sum = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, row_sum, out=np.zeros_like(matrix, dtype=float), where=row_sum > 0)
    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(name)
    for true_id in range(num_classes):
        for pred_id in range(num_classes):
            count = matrix[true_id, pred_id]
            if count == 0:
                continue
            color = "white" if normalized[true_id, pred_id] > 0.5 else "black"
            ax.text(pred_id, true_id, str(count), ha="center", va="center", color=color, fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


def plot_threshold(name, thresholds, scores, chosen):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(thresholds, scores)
    ax.axvline(chosen, color="tomato", linestyle="--", label=f"best {chosen:.2f}")
    ax.set_xlabel("threshold")
    ax.set_ylabel("macro f1")
    ax.set_title(f"{name} threshold")
    ax.legend()
    fig.tight_layout()
    plt.show()


results = {}
for model_name in ("resnet18", "resnet34", "resnet152"):
    print(f"\n=== {model_name} ===", flush=True)
    seed_everything(SEED)
    model = make_model(model_name)
    history = train_model(model, model_name)
    probabilities, y_true = predict(model, val_loader)
    y_pred = probabilities.argmax(axis=1)
    threshold, threshold_f1, thresholds, threshold_scores = best_f1_threshold(probabilities, y_true)
    argmax_f1 = f1_score(y_true, y_pred, **SCORE_ARGS)
    print(
        f"{model_name}  argmax macro f1 {argmax_f1:.3f}  "
        f"best threshold {threshold:.2f}  threshold macro f1 {threshold_f1:.3f}",
        flush=True,
    )
    plot_training(model_name, history)
    plot_confusion(model_name, y_true, y_pred)
    plot_threshold(model_name, thresholds, threshold_scores, threshold)
    results[model_name] = {"threshold": threshold, "f1": threshold_f1, "argmax_f1": argmax_f1}
    model.cpu()
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

winner = max(results, key=lambda name: results[name]["f1"])
print(
    f"best threshold in the whole run: {results[winner]['threshold']:.2f}  "
    f"on {winner}  macro f1 {results[winner]['f1']:.3f}"
)